# Part 1: Python Foundations & Data Structures
### TrendPulse: What is Actually Trending Right Now

**Notebook Discipline & Reproducibility Note:**
- Standard top-to-bottom execution discipline (Runtime -> Restart and run all).
- Only pandas is used to load data into Python list of dictionaries; all downstream operations strictly use core Python standard libraries.

In [ ]:
# Task 1: Setup & Configuration
from datetime import datetime

DATA_DIR = "data"
COUNTRIES = ["US", "GB", "IN"]
VIEW_THRESHOLD = 1_000_000
LIKE_RATIO_HIGH = 0.90
LIKE_RATIO_LOW = 0.70
START_TIME = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("=" * 60)
print(f"RUN HEADER: TrendPulse Pipeline Part 1 | Started: {START_TIME}")
print(f"Target Countries: {COUNTRIES} | Directory: {DATA_DIR}/")
print("=" * 60)

In [ ]:
# Task 2: Loading Function & Categories Mapping
import os, pandas as pd

def load_country(path: str, country: str, max_rows: int = None) -> list:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    df = pd.read_csv(path, nrows=max_rows)
    df["country"] = country
    records = df.to_dict("records")
    print(f"Loaded {len(records):,} records for country: {country}")
    return records

all_records = []
for c in COUNTRIES:
    all_records.extend(load_country(os.path.join(DATA_DIR, f"{c}videos.csv"), c))

print(f"Total raw records loaded: {len(all_records):,}")
cat_df = pd.read_csv(os.path.join(DATA_DIR, "categories.csv"))
category_names = dict(zip(cat_df["category_id"], cat_df["category_name"]))
print(f"Categories mapped: {len(category_names)} categories.")

In [ ]:
# Task 3: Type Conversion and Robust Cleaning
from datetime import datetime

parse_failures = {"views": 0, "likes": 0, "dislikes": 0, "comment_count": 0, "trending_date": 0, "publish_time": 0}
cleaned_records = []

for r in all_records:
    rec = dict(r)
    for field in ["views", "likes", "dislikes", "comment_count"]:
        try:
            rec[field] = int(rec.get(field, 0))
        except (ValueError, TypeError):
            rec[field] = 0
            parse_failures[field] += 1
            
    for flag in ["comments_disabled", "ratings_disabled", "video_error_or_removed"]:
        rec[flag] = True if str(rec.get(flag, False)).strip().lower() in ["true", "1", "t"] else False
        
    try:
        rec["trending_date_dt"] = datetime.strptime(str(rec.get("trending_date", "")).strip(), "%y.%d.%m")
    except (ValueError, TypeError):
        rec["trending_date_dt"] = None
        parse_failures["trending_date"] += 1
        
    try:
        rec["publish_time_dt"] = datetime.strptime(str(rec.get("publish_time", "")).strip()[:19], "%Y-%m-%dT%H:%M:%S")
    except (ValueError, TypeError):
        rec["publish_time_dt"] = None
        parse_failures["publish_time"] += 1
        
    cleaned_records.append(rec)

print("Parsing & Type Cleaning Summary:", parse_failures)

In [ ]:
# Task 4: Core Data Structures
snapshot_tuples = [(r["video_id"], str(r["trending_date"]), r["views"], r["likes"]) for r in cleaned_records[:5]]
print("Immutable Tuples (first 5):", snapshot_tuples)

country_videos = {c: {r["video_id"] for r in cleaned_records if r["country"] == c} for c in COUNTRIES}
for c in COUNTRIES:
    print(f"Unique videos in {c}: {len(country_videos[c]):,}")

multi_country = country_videos["US"].intersection(country_videos["GB"])
print(f"Videos trending in BOTH US & GB: {len(multi_country):,}")
us_only = country_videos["US"].difference(country_videos["GB"], country_videos["IN"])
print(f"Videos unique to US only: {len(us_only):,}")

channel_counts = {}
for r in cleaned_records:
    ch = r.get("channel_title", "Unknown")
    channel_counts[ch] = channel_counts.get(ch, 0) + 1

sorted_channels = sorted(channel_counts.items(), key=lambda x: x[1], reverse=True)
print("\nTop 15 Channels:")
for idx, (ch, cnt) in enumerate(sorted_channels[:15], 1):
    print(f"  {idx:2d}. {ch:30} -> {cnt:,}")

nested_summary = {}
for r in cleaned_records:
    c = r["country"]
    cat = category_names.get(r.get("category_id"), "Unknown")
    if c not in nested_summary:
        nested_summary[c] = {}
    nested_summary[c][cat] = nested_summary[c].get(cat, 0) + 1

print("\nNested Summary Sample:", {c: list(nested_summary[c].items())[:2] for c in COUNTRIES})

In [ ]:
# Task 5: Variable Scope
global_threshold = 500_000

def scope_demo():
    local_var = "Local Scope"
    print(f"Inside function: global_threshold = {global_threshold}, local_var = {local_var}")

scope_demo()
try:
    print(local_var)
except NameError as e:
    print(f"Outside function local access failed: {e}")

In [ ]:
# Task 6: Loops Over Strings and Sequences
def title_features(title: str) -> dict:
    words = str(title).split()
    upper_cnt = sum(1 for w in words if w.isupper() and len(w) > 1)
    punct_cnt = sum(1 for i in range(len(str(title))) if str(title)[i] in ["!", "?"])
    return {"word_count": len(words), "uppercase_words": upper_cnt, "punctuation_marks": punct_cnt}

print("Sample Title Features:", title_features(cleaned_records[0].get("title", "")))

idx = 0
found_video = None
while idx < len(cleaned_records):
    if cleaned_records[idx]["views"] < 10_000_000:
        idx += 1
        continue
    found_video = cleaned_records[idx]
    break

if found_video:
    print(f"First >=10M view video: {found_video.get('title')} ({found_video['views']:,} views)")

In [ ]:
# Task 7: Rule-Based Classification
def classify_video(rec: dict) -> str:
    if rec.get("ratings_disabled", False):
        return "Unrated"
    v = rec.get("views", 0)
    l = rec.get("likes", 0)
    d = rec.get("dislikes", 0)
    tot = l + d
    if tot == 0:
        return "Unrated"
    ratio = l / tot
    if v >= 1_000_000:
        return "Viral Hit" if ratio >= 0.90 else ("Divisive" if ratio < 0.70 else "Steady Performer")
    elif v >= 100_000:
        return "Divisive" if (ratio < 0.70 and tot >= 1_000) else "Steady Performer"
    return "Low Engagement"

label_counts = {}
for r in cleaned_records:
    lbl = classify_video(r)
    label_counts[lbl] = label_counts.get(lbl, 0) + 1

print("Classification Summary:", label_counts)

In [ ]:
# Task 8: Refactored Helper
def safe_divide(num: float, den: float, default: float = 0.0) -> float:
    return num / den if den != 0 else default

print("Safe divide test (100, 0):", safe_divide(100, 0))
print("Safe divide test (90, 100):", safe_divide(90, 100))